# Pearls AQI Predictor — Exploratory Data Analysis
Run this after you have at least a few weeks of data in the feature store
(either via `backfill.py` or after the hourly pipeline has been running for a while).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from feature_store import read_features, DEFAULT_BACKEND

df = read_features(backend=DEFAULT_BACKEND)
print(df.shape)
df.head()

## 1. AQI distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['aqi'].hist(bins=40, ax=axes[0])
axes[0].set_title('AQI distribution')
axes[0].set_xlabel('AQI')

df.boxplot(column='aqi', by='hour', ax=axes[1])
axes[1].set_title('AQI by hour of day')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 2. Day-of-week and monthly patterns
In Peshawar specifically, expect a strong winter spike from crop residue burning
and temperature-inversion trapping of pollutants — worth calling out explicitly
once you have multi-month data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df.groupby('day_of_week')['aqi'].mean().plot(kind='bar', ax=axes[0])
axes[0].set_title('Mean AQI by day of week (0=Mon)')

df.groupby('month')['aqi'].mean().plot(kind='bar', ax=axes[1], color='orange')
axes[1].set_title('Mean AQI by month')
plt.tight_layout()
plt.show()

## 3. Correlation heatmap

In [ ]:
cols = ['aqi', 'pm25', 'pm10', 'o3', 'no2', 'so2', 'co', 'temp', 'humidity', 'pressure', 'wind_speed']
corr = df[cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature correlation with AQI')
plt.tight_layout()
plt.show()

## 4. Missing data audit

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
print('Columns with missing data (fraction):')
print(missing)
missing.plot(kind='barh', figsize=(8, max(3, len(missing)*0.3)))
plt.title('Missing data fraction by column')
plt.tight_layout()
plt.show()

## 5. Autocorrelation of AQI
Confirms how far back lag features should reasonably look.

In [ ]:
from pandas.plotting import autocorrelation_plot
sorted_df = df.sort_values('timestamp')
plt.figure(figsize=(10, 4))
autocorrelation_plot(sorted_df['aqi'].dropna())
plt.title('AQI autocorrelation')
plt.xlim(0, 200)
plt.show()

## 6. Time series view (raw, with rolling mean overlay)

In [ ]:
sorted_df = df.sort_values('timestamp')
plt.figure(figsize=(14, 4))
plt.plot(sorted_df['timestamp'], sorted_df['aqi'], alpha=0.4, label='Hourly AQI')
plt.plot(sorted_df['timestamp'], sorted_df['aqi'].rolling(24).mean(), color='red', label='24h rolling mean')
plt.legend()
plt.title('AQI over time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()